In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from scipy.optimize import minimize


data = pd.read_csv('Task3and4_Loan_Data.csv')
# Extract relevant columns
fico_scores = data['fico_score'].values.reshape(-1, 1) 
defaults = data['default'].values
total_records = np.ones_like(defaults)  


In [ ]:

num_clusters = 10 # Set number of buckets (clusters)



In [ ]:
min_fico = 300
max_fico = 850
number_of_scores = max_fico - min_fico + 1

# Number of borrowers and defaults at each FICO score
records_by_score = np.bincount(
    fico_scores.flatten() - min_fico,
    minlength=number_of_scores
)

defaults_by_score = np.bincount(
    fico_scores.flatten() - min_fico,
    weights=defaults,
    minlength=number_of_scores
)

# Cumulative values allow fast evaluation of each possible bucket
cumulative_records = np.concatenate(
    ([0], np.cumsum(records_by_score))
)

cumulative_defaults = np.concatenate(
    ([0], np.cumsum(defaults_by_score))
)


def log_likelihood_for_clusters(start, end):
    n_i = cumulative_records[end] - cumulative_records[start]
    k_i = cumulative_defaults[end] - cumulative_defaults[start]

    if n_i == 0:
        return -np.inf

    p_i = k_i / n_i

    if p_i == 0 or p_i == 1:
        return 0.0

    return (
        k_i * np.log(p_i)
        + (n_i - k_i) * np.log(1 - p_i)
    )


# Dynamic programming table
dp = np.full(
    (num_clusters + 1, number_of_scores + 1),
    -np.inf
)

previous = np.full(
    (num_clusters + 1, number_of_scores + 1),
    -1,
    dtype=int
)

dp[0, 0] = 0.0

for bucket in range(1, num_clusters + 1):
    for end in range(bucket, number_of_scores + 1):
        for start in range(bucket - 1, end):

            value = (
                dp[bucket - 1, start]
                + log_likelihood_for_clusters(start, end)
            )

            if value > dp[bucket, end]:
                dp[bucket, end] = value
                previous[bucket, end] = start


# Recover the optimal boundaries
cut_indexes = [number_of_scores]
end = number_of_scores

for bucket in range(num_clusters, 0, -1):
    end = previous[bucket, end]
    cut_indexes.append(end)

optimized_boundaries = (
    min_fico + np.array(sorted(cut_indexes))
)

optimized_log_likelihood = dp[
    num_clusters,
    number_of_scores
]

In [ ]:
print(
    "Optimized Cluster Boundaries:",
    optimized_boundaries
)

print(
    "Maximized Log-Likelihood:",
    optimized_log_likelihood
)

bucket_number = pd.cut(
    fico_scores.flatten(),
    bins=optimized_boundaries,
    labels=False,
    include_lowest=True,
    right=False
)

# Rating 1 represents the highest FICO bucket
data['optimized_cluster'] = num_clusters - bucket_number

clusters_default_rate = data.groupby(
    'optimized_cluster'
).agg(
    fico_score_mean=('fico_score', 'mean'),
    default_rate=('default', 'mean'),
    count=('default', 'count')
).sort_index()

print("\nDefault rates per optimized cluster:")
print(clusters_default_rate)